# Volatility Dynamics 10c — long-history physical tail for 0DTE wings

**VD 023.** Module 10b found the wings RICH against realized outcomes, but on only 18 sessions. That leaves a peso problem: one ±8σ day per asset-year would erase the bid-side premium. 10c replaces the outcome pool with about two years of 5-minute history, strictly before the first quote, so the pool is fully out of sample.

**Source.** Binance spot `<ASSET>USDT` closes (index constituents), with the Deribit perpetual as fallback, downloaded by `tools/download_price_history.py`.

**Outcomes.** Built exactly as in 10b:

$$u=\ln(S_T/I_t)/\sqrt{D_t}$$

using the Module 08 clock, backward 4h bipower and settlement weights.

**Clock.** $D$ uses exactly Module 10's clock, so pool outcomes and quotes are standardized identically. A clock refitted on the long history is reported as a sensitivity check.

**Pre-registered (VD 023):**
- **Gate.** On the quote window, the long-history $u$ is compared with the index-based $u$ from 10b, per asset. An asset passes if the correlation is at least 0.90 and the sd ratio lies in [0.85, 1.15]. Failing assets are removed from the pool; if more than 2 assets fail, the verdict is GATE_FAIL.
- **Decision V_bid.** $M$ = median of the session-median $\ln(\text{bid}/P)$ on Module 10's primary rows. The CI comes from a joint bootstrap that resamples quote sessions and history dates (B = 1000), giving RICH, CHEAP or CONSISTENT. **V_mid** is computed the same way for continuity with 10 and 10b.
- **Consequence.** If V_bid is RICH, the next step is a pre-registered, defined-risk wing-selling backtest. Otherwise no wing-selling research from this branch until there are more sessions.

Reported only: net-of-fee bid, per asset, buckets, TTE-conditioned and own-asset pools, tails by volatility tercile, and break-even disaster mass. Outputs: `10c/`.

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_jump_option_wings as ow
from data_engineering.modeling import volatility_wings_empirical as we
from data_engineering.modeling import volatility_wings_longhistory as wl
ASSETS = [x.strip().upper() for x in os.getenv('VD10C_ASSETS', 'AVAX,BTC,ETH,HYPE,SOL,TRX,XRP').split(',') if x.strip()]
HIST_DIR = Path(os.getenv('VD10C_HISTORY_DIR', str(DATA_ROOT / 'history' / 'price_5m')))
N_BOOT = int(os.getenv('VD10C_N_BOOT', str(wl.N_BOOT)))
VD = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics'
OUT = VD / '10c'
OUT.mkdir(parents=True, exist_ok=True)
print('Module 10c', wl.VERSION, wl.selftest())
manifest = HIST_DIR / 'manifest.json'
if manifest.exists():
    print(manifest.read_text())


Module 10c 0.1.0 {'returns_consecutive_only': True, 'true_pool_consistent': True, 'thin_pool_rich': True, 'gate_pass_identical': True, 'gate_fail_scaled': True, 'breakeven_positive_when_rich': True, 'breakeven_zero_when_cheap': True, 'fees_bounded': True}
{
  "AVAX": {
    "status": "ok",
    "source": [
      "binance_spot"
    ],
    "symbol": [
      "AVAXUSDT"
    ],
    "first": "2024-09-01 00:05:00+00:00",
    "last": "2026-10-11 00:05:00+00:00",
    "n_bars": 221761
  },
  "BTC": {
    "status": "ok",
    "source": [
      "binance_spot"
    ],
    "symbol": [
      "BTCUSDT"
    ],
    "first": "2024-09-01 00:05:00+00:00",
    "last": "2026-10-11 00:05:00+00:00",
    "n_bars": 221761
  },
  "ETH": {
    "status": "ok",
    "source": [
      "binance_spot"
    ],
    "symbol": [
      "ETHUSDT"
    ],
    "first": "2024-09-01 00:05:00+00:00",
    "last": "2026-10-11 00:05:00+00:00",
    "n_bars": 221761
  },
  "HYPE": {
    "status": "ok",
    "source": [
      "binance_spot"
  

In [2]:
t0 = time.time()
hist = wl.load_history(HIST_DIR, ASSETS)
print(hist.groupby('underlying').agg(first=('state_time', 'min'), last=('state_time', 'max'), n=('close', 'size'),
                                     source=('source', lambda s: ','.join(sorted(set(s))))))
frame = wl.return_frame(hist)
from data_engineering.modeling import volatility_vd_b_core as vb
con = duckdb.connect(str(DB_PATH), read_only=True)
dbf = con.execute("""SELECT state_time, underlying, return_5m FROM time_series_feature_state
                     WHERE return_5m IS NOT NULL""").df()
con.close()
dbf = dbf[dbf.underlying.isin(ASSETS) & (pd.to_datetime(dbf.state_time, utc=True) < ow.VRP_CONFIRMATION_START)]
clock = wl.module10_clock(vb.build_grid(dbf))
oc = wl.long_outcomes(frame, clock=clock)
oc_refit = wl.long_outcomes(frame)
print(f'{len(oc):,} long-history origins in {time.time()-t0:.0f}s')
q = pd.read_csv(VD / '10' / '10_priced_quotes.csv.gz')
o10b = pd.read_csv(VD / '10b' / '10b_origin_outcomes.csv.gz')
pool_all, overlap = wl.split(oc, sorted(q.session.unique()))
gate = wl.consistency_gate(overlap, o10b)
display(gate.round(3))
pool_refit, overlap_refit = wl.split(oc_refit, sorted(q.session.unique()))
gate_refit = wl.consistency_gate(overlap_refit, o10b)
print('sensitivity: clock refitted on long history')
display(gate_refit.round(3))
failed = gate.loc[~gate['pass'], 'underlying'].tolist()
gate_fail = len(failed) > wl.GATE_MAX_FAILS
pool = pool_all[~pool_all.underlying.isin(failed)].reset_index(drop=True)
print('gate failures:', failed, '| GATE_FAIL:', gate_fail)
print(f'pool: {len(pool):,} origins, {pool.session.nunique()} dates, {pool.session.min()} -> {pool.session.max()}')


                               first                      last       n        source
underlying                                                                          
AVAX       2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
BTC        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
ETH        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
HYPE       2026-09-24 11:05:00+00:00 2026-10-11 00:05:00+00:00    4765  binance_spot
SOL        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
TRX        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
XRP        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
1,060,459 long-history origins in 287s


,underlying,n_matched,corr,sd_ratio,pass
0,AVAX,3998,0.999,0.999,True
1,BTC,3998,0.997,1.026,True
2,ETH,3998,0.997,0.963,True
3,HYPE,3314,0.999,0.979,True
4,SOL,3998,0.999,0.984,True
5,TRX,3998,0.974,1.007,True
6,XRP,3998,0.998,0.979,True


sensitivity: clock refitted on long history


,underlying,n_matched,corr,sd_ratio,pass
0,AVAX,3998,0.997,1.016,True
1,BTC,3998,0.989,1.052,True
2,ETH,3998,0.987,0.998,True
3,HYPE,3314,0.990,0.896,True
4,SOL,3998,0.995,1.014,True
5,TRX,3998,0.955,0.994,True
6,XRP,3998,0.992,1.017,True


gate failures: [] | GATE_FAIL: False
pool: 1,027,968 origins, 749 dates, 2024-09-01 -> 2026-09-19


In [3]:
excluded = json.loads((VD / '10' / '10_decision.json').read_text()).get('excluded_assets_tick_artifact', [])
prim = ow.primary_sample(q, excluded).reset_index(drop=True)
prim['net_bid_usd'] = wl.net_bid(prim)
t0 = time.time()
dec = wl.decide(prim, pool, n_boot=N_BOOT, price_cols=('bid_usd', 'mid_usd', 'net_bid_usd'), gate_fail=gate_fail)
print(f'joint bootstrap ({time.time()-t0:.0f}s)')
print(json.dumps(dec, indent=2, default=float))


joint bootstrap (834s)
{
  "n_rows": 14805,
  "n_sessions": 19,
  "n_pool_origins": 1027968,
  "n_pool_dates": 749,
  "share_model_price_zero": 0.0,
  "bid_usd": {
    "M": -0.27864792410499667,
    "ci_low": -0.5848589664236828,
    "ci_high": 0.2698138638336553,
    "verdict": "CONSISTENT",
    "bootstrap_verdict": "CONSISTENT",
    "n_boot": 1000
  },
  "mid_usd": {
    "M": 0.2550503913884161,
    "ci_low": -0.049111225720173686,
    "ci_high": 0.7091774711895895,
    "verdict": "CONSISTENT",
    "bootstrap_verdict": "CONSISTENT",
    "n_boot": 1000
  },
  "net_bid_usd": {
    "M": -0.41217931672951924,
    "ci_low": -0.7183903590482055,
    "ci_high": 0.13628247120913276,
    "verdict": "CONSISTENT",
    "bootstrap_verdict": "CONSISTENT",
    "n_boot": 1000
  }
}


In [4]:
P = wl.price(prim, pool.u.to_numpy())
by_asset = pd.DataFrame([{'underlying': u, 'price': c, **wl.quote_only(g, P[g.index], c)}
                         for u, g in prim.groupby('underlying') for c in ('bid_usd', 'mid_usd')])
display(by_asset.round(3))
own = []
for u, g in prim.groupby('underlying'):
    pu = pool[pool.underlying == u]
    if len(pu) < 1000:
        continue
    Pu = wl.price(g, pu.u.to_numpy())
    own.append({'underlying': u, **wl.quote_only(g, Pu, 'bid_usd')})
own = pd.DataFrame(own)
display(own.round(3))
tte = []
for tb, g in prim.groupby('tte_bucket'):
    pt = pool[pool.tte_bucket == tb]
    Pt = wl.price(g, pt.u.to_numpy())
    tte.append({'tte_bucket': tb, **{f'{c}_{k}': v for c in ('bid_usd', 'mid_usd')
                                     for k, v in wl.quote_only(g, Pt, c).items() if k in ('M', 'ci_low', 'ci_high', 'verdict', 'n_rows')}})
tte = pd.DataFrame(tte)
display(tte.round(3))


,underlying,price,n_rows,n_sessions,M,ci_low,ci_high,verdict,mean_ratio_mid_over_model,share_model_price_zero
0,AVAX,bid_usd,3878,19,-1.117,-1.742,-0.744,CHEAP,0.756,0.0
1,AVAX,mid_usd,3878,19,-0.348,-0.635,-0.017,CHEAP,1.551,0.0
2,BTC,bid_usd,2704,19,-0.302,-0.472,0.170,CONSISTENT,1.360,0.0
3,BTC,mid_usd,2704,19,0.169,0.050,0.590,RICH,2.152,0.0
4,ETH,bid_usd,3286,19,-0.003,-0.221,0.527,CONSISTENT,2.077,0.0
5,ETH,mid_usd,3286,19,0.349,0.239,0.843,RICH,2.943,0.0
6,HYPE,bid_usd,2451,19,-1.019,-1.332,-0.857,CHEAP,0.922,0.0
7,HYPE,mid_usd,2451,19,-0.093,-0.259,0.362,CONSISTENT,4.308,0.0
8,SOL,bid_usd,79,10,1.025,0.556,1.265,RICH,3.245,0.0
9,SOL,mid_usd,79,10,1.248,0.779,1.453,RICH,4.056,0.0


,underlying,n_rows,n_sessions,M,ci_low,ci_high,verdict,mean_ratio_mid_over_model,share_model_price_zero
0,AVAX,3878,19,-1.142,-1.747,-0.832,CHEAP,0.723,0.0
1,BTC,2704,19,-0.443,-0.635,-0.008,CHEAP,1.093,0.0
2,ETH,3286,19,-0.162,-0.396,0.267,CONSISTENT,1.647,0.0
3,SOL,79,10,1.271,0.824,1.560,RICH,4.550,0.0
4,TRX,1658,19,1.343,0.938,1.536,RICH,10.392,0.0
5,XRP,749,19,0.468,0.120,1.021,RICH,3.371,0.0


,tte_bucket,bid_usd_n_rows,bid_usd_M,bid_usd_ci_low,bid_usd_ci_high,bid_usd_verdict,mid_usd_n_rows,mid_usd_M,mid_usd_ci_low,mid_usd_ci_high,mid_usd_verdict
0,"(120.0, 240.0]",375,0.651,0.100,1.042,RICH,375,1.201,0.835,2.035,RICH
1,"(240.0, 480.0]",1678,0.796,0.614,1.089,RICH,1678,1.439,1.180,1.700,RICH
2,"(480.0, 960.0]",6638,-0.431,-0.868,-0.026,CHEAP,6638,0.138,-0.176,0.556,CONSISTENT
3,"(60.0, 120.0]",69,-0.088,-0.661,0.183,CONSISTENT,69,0.703,0.355,0.883,RICH
4,"(960.0, 1200.0]",6037,-0.883,-1.014,-0.450,CHEAP,6037,-0.411,-0.567,-0.112,CHEAP


In [5]:
allq = q[(~q.floor_bound) & (~q.underlying.isin(excluded))].reset_index(drop=True)
Pa = wl.price(allq, pool.u.to_numpy())
rows = []
for (zb, tb), g in allq.groupby(['z_bucket', 'tte_bucket']):
    for c in ('bid_usd', 'mid_usd'):
        r = wl.quote_only(g, Pa[g.index], c, n_boot=500)
        rows.append({'z_bucket': zb, 'tte_bucket': tb, 'price': c, **{k: r.get(k) for k in ('n_rows', 'n_sessions', 'M', 'ci_low', 'ci_high', 'verdict', 'share_model_price_zero')}})
buckets = pd.DataFrame(rows)
display(buckets[buckets.price == 'bid_usd'].pivot(index='z_bucket', columns='tte_bucket', values='M').round(3))
tails = we.tail_table(pool)
display(tails.round(4))
vt = wl.vol_tercile_tails(pool)
display(vt.round(4))
be = pd.DataFrame([{'u_star': us, 'price': c, 'breakeven_p_per_origin': wl.breakeven(prim, pool.u.to_numpy(), c, us),
                    'observed_share_beyond': float(np.mean(np.abs(pool.u) > us))}
                   for us in wl.BREAKEVEN_USTAR for c in ('bid_usd', 'mid_usd')])
display(be)
pr = pool_refit[~pool_refit.underlying.isin(failed)]
refit_sens = {c: wl.quote_only(prim, wl.price(prim, pr.u.to_numpy()), c) for c in ('bid_usd', 'mid_usd')}
print('sensitivity, refitted clock pool:', json.dumps(refit_sens, indent=2, default=float))


tte_bucket,"(120.0, 240.0]","(240.0, 480.0]","(480.0, 960.0]","(60.0, 120.0]","(960.0, 1200.0]"
z_bucket,,,,,
"[0.0, 1.0)",-0.396,-0.272,-0.266,-0.420,-0.150
"[1.0, 2.0)",-0.552,-0.313,-0.661,-0.478,-0.366
"[2.0, 3.0)",-0.417,-0.198,-0.637,-0.380,-0.535
"[3.0, 5.0)",0.794,0.527,-0.079,0.834,0.479
"[5.0, 8.0)",2.185,2.116,2.189,NaN,1.845
"[8.0, inf)",3.759,3.251,3.249,4.778,2.161


,group,n_origins,n_session_assets,u_mean,u_sd,log_k_mean,log_k_sd,P_abs_u_gt_1,normal_1,distinct_session_assets_gt_1,P_abs_u_gt_2,normal_2,distinct_session_assets_gt_2,P_abs_u_gt_3,normal_3,distinct_session_assets_gt_3,P_abs_u_gt_4,normal_4,distinct_session_assets_gt_4,P_abs_u_gt_5,normal_5,distinct_session_assets_gt_5
0,ALL,1027968,4494,0.0028,1.3118,0.1232,0.8227,0.3130,0.3173,4374,0.0938,0.0455,2389,0.0356,0.0027,1059,0.0160,0.0001,497,0.0075,0.0,261
1,AVAX,171328,749,-0.0011,1.2942,0.1192,0.7721,0.3129,0.3173,729,0.0954,0.0455,401,0.0353,0.0027,168,0.0169,0.0001,78,0.0076,0.0,47
2,BTC,171328,749,0.0034,1.3542,0.1351,0.8933,0.3054,0.3173,735,0.0958,0.0455,400,0.0407,0.0027,190,0.0187,0.0001,89,0.0092,0.0,54
3,ETH,171328,749,0.0120,1.3593,0.1200,0.8964,0.3012,0.3173,723,0.0951,0.0455,382,0.0405,0.0027,182,0.0205,0.0001,96,0.0101,0.0,57
4,SOL,171328,749,0.0128,1.2273,0.0747,0.7729,0.3049,0.3173,735,0.0855,0.0455,392,0.0281,0.0027,155,0.0116,0.0001,63,0.0053,0.0,29
5,TRX,171328,749,0.0358,1.3784,0.2311,0.7425,0.3640,0.3173,726,0.1090,0.0455,444,0.0405,0.0027,208,0.0160,0.0001,105,0.0070,0.0,46
6,XRP,171328,749,-0.0460,1.2483,0.0591,0.8346,0.2895,0.3173,726,0.0821,0.0455,370,0.0284,0.0027,156,0.0124,0.0001,66,0.0060,0.0,28
7,"TTE (120.0, 240.0]",107850,4494,-0.0244,1.1520,-0.0269,0.7086,0.2907,0.3173,2694,0.0781,0.0455,803,0.0236,0.0027,268,0.0081,0.0001,96,0.0033,0.0,42
8,"TTE (240.0, 480.0]",215424,4488,0.0173,1.0352,-0.2108,0.7189,0.2539,0.3173,2677,0.0590,0.0455,759,0.0173,0.0027,241,0.0054,0.0001,89,0.0013,0.0,32
9,"TTE (480.0, 960.0]",430848,4488,0.0034,1.3163,0.1251,0.8380,0.3098,0.3173,3137,0.0920,0.0455,1107,0.0354,0.0027,447,0.0162,0.0001,207,0.0080,0.0,113


,vol_tercile,n_origins,u_sd,P_abs_u_gt_1,P_abs_u_gt_2,P_abs_u_gt_3,P_abs_u_gt_4,P_abs_u_gt_5
0,high,342654,0.9630,0.2299,0.0462,0.0124,0.0035,0.0012
1,low,342660,1.6463,0.3973,0.1501,0.0649,0.0329,0.0163
2,mid,342654,1.2348,0.3116,0.0852,0.0295,0.0116,0.0051


,u_star,price,breakeven_p_per_origin,observed_share_beyond
0,6.0,bid_usd,0.000000,0.003849
1,6.0,mid_usd,0.005543,0.003849
2,8.0,bid_usd,0.000000,0.001324
3,8.0,mid_usd,0.003197,0.001324
4,10.0,bid_usd,0.000000,0.000518
5,10.0,mid_usd,0.002294,0.000518


sensitivity, refitted clock pool: {
  "bid_usd": {
    "n_rows": 14805,
    "n_sessions": 19,
    "M": -0.17309902698581975,
    "ci_low": -0.4128620813440594,
    "ci_high": 0.2153037089764142,
    "verdict": "CONSISTENT",
    "mean_ratio_mid_over_model": 2.2998391349195093,
    "share_model_price_zero": 0.0
  },
  "mid_usd": {
    "n_rows": 14805,
    "n_sessions": 19,
    "M": 0.35405645489496984,
    "ci_low": 0.10455959369934675,
    "ci_high": 0.6826110081743569,
    "verdict": "RICH",
    "mean_ratio_mid_over_model": 4.080290591331435,
    "share_model_price_zero": 0.0
  }
}


In [6]:
gate.to_csv(OUT / '10c_consistency_gate.csv', index=False)
gate_refit.to_csv(OUT / '10c_consistency_gate_refit_clock.csv', index=False)
pd.DataFrame([{'price': c, **v} for c, v in refit_sens.items()]).to_csv(OUT / '10c_refit_clock_sensitivity.csv', index=False)
by_asset.to_csv(OUT / '10c_by_asset.csv', index=False)
own.to_csv(OUT / '10c_own_asset_pool_bid.csv', index=False)
tte.to_csv(OUT / '10c_tte_conditioned.csv', index=False)
buckets.to_csv(OUT / '10c_bucket_table.csv', index=False)
tails.to_csv(OUT / '10c_tail_frequencies.csv', index=False)
vt.to_csv(OUT / '10c_tails_by_vol_tercile.csv', index=False)
be.to_csv(OUT / '10c_breakeven_disaster.csv', index=False)
if os.getenv('VD10C_EXPORT_POOL', '0') == '1':      # ~50 MB; off by default to keep the repo small
    pool.drop(columns=['bar_index']).to_csv(OUT / '10c_pool_outcomes.csv.gz', index=False, compression='gzip')
pool.groupby(['underlying', 'session']).u.agg(['size', 'min', 'max', 'std']).reset_index().to_csv(
    OUT / '10c_pool_session_summary.csv.gz', index=False, compression='gzip')
decision = {'V_bid': dec['bid_usd'], 'V_mid': dec['mid_usd'], 'net_bid_reported': dec['net_bid_usd'],
            'gate_failures': failed, 'gate_fail': gate_fail, 'pool': {k: dec[k] for k in ('n_pool_origins', 'n_pool_dates')},
            'share_model_price_zero': dec['share_model_price_zero']}
(OUT / '10c_decision.json').write_text(json.dumps(decision, indent=2, default=float))
meta = {'version': wl.VERSION, 'module': '10c_long_history_wing_benchmark', 'history_dir': str(HIST_DIR),
        'sources': sorted(set(hist.source)), 'pool_last_session': wl.POOL_LAST_SESSION, 'n_boot': N_BOOT,
        'tail_raw_share': wl.TAIL_RAW_LONG, 'consumes': ['10/10_priced_quotes.csv.gz', '10b/10b_origin_outcomes.csv.gz']}
(OUT / '10c_run_metadata.json').write_text(json.dumps(meta, indent=2, default=float))
print(json.dumps(decision, indent=2, default=float))


{
  "V_bid": {
    "M": -0.27864792410499667,
    "ci_low": -0.5848589664236828,
    "ci_high": 0.2698138638336553,
    "verdict": "CONSISTENT",
    "bootstrap_verdict": "CONSISTENT",
    "n_boot": 1000
  },
  "V_mid": {
    "M": 0.2550503913884161,
    "ci_low": -0.049111225720173686,
    "ci_high": 0.7091774711895895,
    "verdict": "CONSISTENT",
    "bootstrap_verdict": "CONSISTENT",
    "n_boot": 1000
  },
  "net_bid_reported": {
    "M": -0.41217931672951924,
    "ci_low": -0.7183903590482055,
    "ci_high": 0.13628247120913276,
    "verdict": "CONSISTENT",
    "bootstrap_verdict": "CONSISTENT",
    "n_boot": 1000
  },
  "gate_failures": [],
  "gate_fail": false,
  "pool": {
    "n_pool_origins": 1027968,
    "n_pool_dates": 749
  },
  "share_model_price_zero": 0.0
}
